# Proyek Akhir — Machine Learning Operations (MLOps)

## Sistem Machine Learning End-to-End: Prediksi Usia Abalon

**Username Dicoding:** sonnyariady

**Kelas:** Machine Learning Operations (MLOps) — Dicoding

Proyek ini membangun sistem machine learning end-to-end:

1. **Machine learning pipeline** menggunakan TensorFlow Extended (TFX) yang diorkestrasikan dengan **Apache Beam**.
2. **Deployment** model ke environment cloud (Heroku/Railway).
3. **Monitoring** sistem menggunakan **Prometheus** (dan Grafana).

Seluruh tahapan proyek dijelaskan pada *text cell* pada setiap bagian notebook ini, sesuai ketentuan dokumentasi proyek.

## 1. Informasi Dataset

Dataset yang digunakan adalah **Abalone Data Set** dari UCI Machine Learning Repository:

- **Sumber:** https://archive.ics.uci.edu/ml/datasets/abalone
- **Jumlah sampel:** 4.177 baris
- **Format:** CSV (tanpa header; nama kolom ditambahkan pada tahap pengolahan data)

Dataset berisi pengukuran fisik cangkang abalon (kerang laut) yang digunakan untuk memprediksi usianya. Deskripsi kolom:

| Kolom | Tipe | Keterangan |
|---|---|---|
| Sex | Kategorikal | Jenis kelamin: M (jantan), F (betina), I (muda) |
| Length | Numerik | Panjang cangkang (mm) |
| Diameter | Numerik | Diameter cangkang (mm) |
| Height | Numerik | Tinggi cangkang (mm) |
| Whole weight | Numerik | Berat total abalon (gram) |
| Shucked weight | Numerik | Berat daging (gram) |
| Viscera weight | Numerik | Berat organ dalam (gram) |
| Shell weight | Numerik | Berat cangkang (gram) |
| Rings | Numerik | Jumlah cincin (1 cincin ± 1,5 tahun usia) |
| label | Numerik (biner) | Hasil rekayasa data: 1 jika Rings > 9 (dewasa), 0 jika muda |

**Rekayasa label:** umur abalon (tahun) ± Rings + 1,5. Abalon dengan Rings > 9 (usia > 10,5 tahun) diberi label `1` (dewasa) dan sisanya `0` (muda). Label dibuat pada tahap pengolahan data di modul `modules/data_processing.py`.

## 2. Persoalan Bisnis yang Ingin Diselesaikan

Usia abalon merupakan indikator penting dalam industri perikanan dan budidaya kerang — menentukan waktu panen optimal, pengelolaan stok, hingga penetapan harga jual. Sayangnya, penentuan usia abalon secara akurat hanya bisa dilakukan dengan menghitung jumlah cincin pada cangkang, yang membutuhkan pembedahan dan pemeriksaan mikroskop.

**Persoalan:** membangun sistem yang dapat memprediksi kategori usia abalon (dewasa/muda) secara otomatis hanya dari pengukuran fisik yang mudah diperoleh (panjang, berat, dan sebagainya) tanpa harus membedah cangkang.

**Stakeholder:** pembudidaya kerang, pengelola perikanan, dan peneliti kelautan.

## 3. Solusi Machine Learning & Target

**Solusi:** sistem **klasifikasi biner** dengan *deep neural network* (DNN) yang dilatih menggunakan pipeline TFX:

- **Input:** 8 fitur fisik (1 kategorikal + 7 numerik).
- **Output:** probabilitas abalon termasuk kategori dewasa (label 1).

**Target yang ingin dicapai:**

- Akurasi (*BinaryAccuracy*) pada data evaluasi ≥ 75%.
- AUC ≥ 0,80.
- Precision dan Recall yang seimbang.

**Alasan pemilihan dataset:** dataset tabular berukuran kecil-menengah sangat cocok untuk mendemonstrasikan seluruh tahapan MLOps (pipeline, deployment, monitoring) dengan waktu training yang wajar.

## 4. Persiapan Lingkungan

Notebook ini dijalankan menggunakan **Python 3.10** dengan dependency pada `requirements.txt`:

```bash
python -m venv venv
source venv/bin/activate        # Linux/macOS/Git Bash
pip install -r requirements.txt
```

Struktur proyek:

- `sonnyariady-pipeline/` — seluruh komponen pipeline TFX (ExampleGen s.d. Pusher)
- `modules/` — modul reusable (penerapan clean code)
- `data/` — dataset abalone
- `pipeline_output/` — artifact hasil eksekusi pipeline
- `serving_model/` — model serving hasil komponen Pusher
- `monitoring/` — konfigurasi Prometheus & Grafana

In [1]:
import os
import sys

PROJECT_ROOT = os.getcwd()
for path in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "sonnyariady-pipeline")):
    if path not in sys.path:
        sys.path.insert(0, path)

import tensorflow as tf
from tfx import v1 as tfx

print("Python version :", sys.version.split()[0])
print("TensorFlow     :", tf.__version__)
print("TFX            :", tfx.__version__)

C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\google\auth\transport\grpc.py:44: FutureWarning: grpcio < 1.83.0 does not support Post-Quantum Cryptography (PQC). Support for non-PQC environments is deprecated. In October 2026, google-auth will raise its minimum requirements to enforce grpcio >= 1.83.0. For more details on Google Cloud's post-quantum security migration, visit: https://cloud.google.com/security/resources/post-quantum-cryptography
  warnings.warn(
C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\venv\lib\site-packages\g

Python version : 3.10.0
TensorFlow     : 2.13.1
TFX            : 1.14.0


In [2]:
import pandas as pd

from modules.data_processing import LABEL_KEY, prepare_dataset
from modules.utils import set_seed

set_seed()

# Unduh & siapkan dataset (otomatis dilewati jika file sudah tersedia).
data_file = prepare_dataset(os.path.join(PROJECT_ROOT, "data"))

frame = pd.read_csv(data_file)
print("Jumlah baris :", len(frame))
print("Jumlah kolom :", len(frame.columns))
print()
print("Distribusi label (dewasa/muda):")
print(frame[LABEL_KEY].value_counts(normalize=True).sort_index())
print()
display(frame.head())

Jumlah baris : 4177
Jumlah kolom : 10

Distribusi label (dewasa/muda):
0    0.501796
1    0.498204
Name: label, dtype: float64



,Sex,Length,Diameter,Height,Whole weight,Shucked weight,Viscera weight,Shell weight,Rings,label
0,M,0.605,0.455,0.160,1.1035,0.4210,0.3015,0.325,9,0
1,M,0.590,0.440,0.150,0.8725,0.3870,0.2150,0.245,8,0
2,F,0.560,0.445,0.195,0.9810,0.3050,0.2245,0.335,16,1
3,F,0.635,0.490,0.170,1.2615,0.5385,0.2665,0.380,9,0
4,M,0.475,0.385,0.145,0.6175,0.2350,0.1080,0.215,14,1


## 5. Komponen 1 — ExampleGen

**ExampleGen** mengubah data mentah (CSV) menjadi contoh (*tf.Example*) untuk training dan evaluasi. Pada proyek ini data dibagi menjadi **80% train** dan **20% eval** menggunakan *hash bucket* sehingga pembagiannya deterministik.

In [3]:
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext
from tfx.proto import trainer_pb2

from components import (
    create_evaluator,
    create_example_gen,
    create_example_validator,
    create_pusher,
    create_resolver,
    create_schema_gen,
    create_statistics_gen,
    create_trainer,
    create_transform,
    create_tuner,
)
from configs import SERVING_MODEL_DIR

interactive_root = os.path.abspath('C:/tfx_output')
os.makedirs(interactive_root, exist_ok=True)

context = InteractiveContext(
    pipeline_name="abalone_interactive",
    pipeline_root=interactive_root,
    metadata_connection_config=tfx.orchestration.metadata.sqlite_metadata_connection_config(
        os.path.join(interactive_root, "metadata.sqlite")
    ),
)

example_gen = create_example_gen(os.path.join(PROJECT_ROOT, "data"))
context.run(example_gen, enable_cache=False)

for artifact in example_gen.outputs["examples"].get():
    print("Split :", artifact.split_names, "->", artifact.uri)

Split : ["train", "eval"] -> C:\tfx_output\CsvExampleGen\examples\43


## 6. Komponen 2 — StatisticsGen

**StatisticsGen** menghitung statistik data (distribusi nilai, mean, min, max, dan sebagainya) untuk setiap split. Statistik ini menjadi dasar pembuatan skema dan deteksi anomali.

In [4]:
statistics_gen = create_statistics_gen(example_gen.outputs["examples"])
context.run(statistics_gen)
context.show(statistics_gen.outputs["statistics"])

## 7. Komponen 3 — SchemaGen

**SchemaGen** menginferensi skema data secara otomatis dari statistik: tipe data setiap kolom, domain nilai, dan properti lainnya. Skema ini digunakan komponen-komponen selanjutnya (ExampleValidator, Transform, Trainer).

In [5]:
schema_gen = create_schema_gen(statistics_gen.outputs["statistics"])
context.run(schema_gen)
context.show(schema_gen.outputs["schema"])

,Type,Presence,Valency,Domain
Feature name,,,,
'Diameter',FLOAT,required,single,-
'Height',FLOAT,required,single,-
'Length',FLOAT,required,single,-
'Rings',INT,required,single,-
'Sex',STRING,required,single,'Sex'
'Shell weight',FLOAT,required,single,-
'Shucked weight',FLOAT,required,single,-
'Viscera weight',FLOAT,required,single,-
'Whole weight',FLOAT,required,single,-


,Values
Domain,
'Sex',"'F', 'I', 'M'"


## 8. Komponen 4 — ExampleValidator

**ExampleValidator** membandingkan statistik data terhadap skema untuk mendeteksi anomali, misalnya nilai di luar domain, tipe data tidak sesuai, atau perubahan distribusi. Jika tidak ditemukan anomali, data dinyatakan layak diproses lebih lanjut.

In [6]:
example_validator = create_example_validator(
    statistics_gen.outputs["statistics"], schema_gen.outputs["schema"]
)
context.run(example_validator)
context.show(example_validator.outputs["anomalies"])

## 9. Komponen 5 — Transform

**Transform** menerapkan rekayasa fitur menggunakan **tf.Transform** dengan kode yang sama saat training maupun serving (mencegah *training-serving skew*):

- `Sex` → indeks vocabulary (dengan bucket out-of-vocabulary).
- 7 fitur numerik → standarisasi z-score.
- `label` → diteruskan untuk kebutuhan training/evaluasi.

Modul transform disimpan pada `sonnyariady-pipeline/transform.py`.

In [7]:
import tensorflow_transform as tft

transform = create_transform(example_gen.outputs["examples"], schema_gen.outputs["schema"])
context.run(transform)

transform_graph = tft.TFTransformOutput(transform.outputs["transform_graph"].get()[0].uri)
print("Fitur hasil transformasi:", sorted(transform_graph.transformed_feature_spec().keys()))

INFO:tensorflow:Assets written to: C:\tfx_output\Transform\transform_graph\47\.temp_path\tftransform_tmp\8f5ff16f61b944ca9297acde2c995a5a\assets


INFO:tensorflow:Assets written to: C:\tfx_output\Transform\transform_graph\47\.temp_path\tftransform_tmp\8f5ff16f61b944ca9297acde2c995a5a\assets


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


...

INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


Fitur hasil transformasi: ['Diameter_xf', 'Height_xf', 'Length_xf', 'Sex_xf', 'Shell weight_xf', 'Shucked weight_xf', 'Viscera weight_xf', 'Whole weight_xf', 'label_xf']


## 10. Komponen 6 — Tuner (Saran Penilaian: Hyperparameter Tuning)

**Tuner** menjalankan **hyperparameter tuning otomatis** menggunakan KerasTuner (RandomSearch) untuk mencari kombinasi terbaik:

- learning rate (log-uniform, 1e-4 s.d. 1e-2)
- jumlah unit hidden layer 1 (16 s.d. 128)
- jumlah unit hidden layer 2 (8 s.d. 64)
- dropout (0,0 s.d. 0,5)

Metrik yang dioptimalkan: `val_accuracy`. Hasil hyperparameter terbaik kemudian digunakan oleh Trainer.

In [8]:
train_args = trainer_pb2.TrainArgs(splits=['train'], num_steps=52)
eval_args = trainer_pb2.EvalArgs(splits=['eval'], num_steps=13)

tuner = create_tuner(
    example_gen.outputs["examples"],
    transform.outputs["transform_graph"],
    train_args,
    eval_args,
)

tuner_succeeded = False
try:
    context.run(tuner)
    tuner_succeeded = True
    best_hp_path = os.path.join(
        tuner.outputs["best_hyperparameters"].get()[0].uri,
        "best_hyperparameters.txt",
    )
    with open(best_hp_path, "r", encoding="utf-8") as file:
        print("Hyperparameter terbaik:\n", file.read())
except Exception as exc:  # noqa: BLE001
    print("Tuner tidak dapat dijalankan secara interaktif:", exc)

Trial 5 Complete [00h 00m 06s]
val_accuracy: 0.0

Best val_accuracy So Far: 0.0
Total elapsed time: 00h 00m 31s
Results summary
Results in C:\tfx_output\.temp\48\abalone_hyperparameter_tuning
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 0 summary
Hyperparameters:
learning_rate: 0.00544853401907757
units_1: 112
units_2: 24
dropout: 0.30000000000000004
Score: 0.0

Trial 1 summary
Hyperparameters:
learning_rate: 0.000509655304274329
units_1: 80
units_2: 40
dropout: 0.2
Score: 0.0

Trial 2 summary
Hyperparameters:
learning_rate: 0.0052040269557859075
units_1: 64
units_2: 56
dropout: 0.1
Score: 0.0

Trial 3 summary
Hyperparameters:
learning_rate: 0.001038536674208187
units_1: 32
units_2: 8
dropout: 0.4
Score: 0.0

Trial 4 summary
Hyperparameters:
learning_rate: 0.0019295941277227592
units_1: 32
units_2: 32
dropout: 0.2
Score: 0.0
Hyperparameter terbaik:
 {"space": [{"class_name": "Float", "config": {"name": "learning_rate", "default": 0.0001, "conditions": [

## 11. Komponen 7 — Trainer

**Trainer** melatih model **DNN** pada fitur hasil transformasi:

- Embedding untuk fitur `Sex` (8 dimensi).
- 7 fitur numerik terstandarisasi (z-score).
- 2 hidden layer (Dense + Dropout).
- Output sigmoid (probabilitas abalon dewasa).

Model dikompilasi dengan optimizer Adam, loss `binary_crossentropy`, serta metrik `accuracy` dan `AUC`. Apabila Tuner berhasil dijalankan, hyperparameter terbaiknya digunakan. Hasil training diekspor sebagai **SavedModel** dengan signature `serving_default` yang menerima serialized `tf.Example` (modul: `sonnyariady-pipeline/trainer.py`).

In [9]:
trainer = create_trainer(
    example_gen.outputs["examples"],
    transform.outputs["transform_graph"],
    schema_gen.outputs["schema"],
    train_args,
    eval_args,
    tuner=tuner if tuner_succeeded else None,
)
context.run(trainer)

Epoch 1/20
52/52 [==============================] - 4s 22ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_auc: 0.0000e+00
Epoch 2/20
52/52 [==============================] - 1s 14ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_auc: 0.0000e+00
Epoch 3/20
52/52 [==============================] - 1s 14ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_auc: 0.0000e+00
Epoch 4/20
52/52 [==============================] - 1s 15ms/step - loss: 0.0000e+00 - accuracy: 0.0000e+00 - auc: 0.0000e+00 - val_loss: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_auc: 0.0000e+00
INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: C:\tfx_output\Trainer\model\49\Format-Serving\assets


INFO:tensorflow:Assets written to: C:\tfx_output\Trainer\model\49\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 49
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={}, _input_trigger=None
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}, _input_trigger=None)

## 12. Komponen 8 — Resolver

**Resolver** memilih model terbaik untuk dievaluasi. Strategi `LatestBlessedModelResolver` memilih model terbaru yang telah dinyatakan **blessed** (lolos) oleh Evaluator; jika belum ada model yang blessed, model terbaru yang dipilih.

In [10]:
resolver = create_resolver()
context.run(resolver)

for artifact in resolver.outputs["model"].get():
    print("Model terpilih:", artifact.uri)

## 13. Komponen 9 — Evaluator

**Evaluator** menghitung metrik performa model pada data evaluasi menggunakan **TensorFlow Model Analysis (TFMA)**:

- ExampleCount
- BinaryAccuracy (akurasi)
- AUC
- Precision
- Recall

Model yang memenuhi ambang evaluasi akan diberi status *blessed* dan siap dipublikasikan oleh Pusher.

In [11]:
evaluator = create_evaluator(
    example_gen.outputs["examples"], trainer, resolver, schema_gen.outputs["schema"]
)
context.run(evaluator)
context.show(evaluator.outputs["evaluation"])

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


SlicingMetricsViewer(config={'weightedExamplesColumn': 'example_count'}, data=[{'slice': 'Overall', 'metrics':…

## 14. Komponen 10 — Pusher

**Pusher** mem-publish model yang lolos evaluasi ke direktori `serving_model/` dalam format **SavedModel**. Model inilah yang kemudian dilayani oleh web app (Flask) di environment cloud.

In [12]:
pusher = create_pusher(trainer, evaluator, SERVING_MODEL_DIR)
context.run(pusher)

import glob

print("Model serving tersedia di:", SERVING_MODEL_DIR)
for path in glob.glob(os.path.join(SERVING_MODEL_DIR, "*")):
    print(" -", path)

Model serving tersedia di: C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\assets
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\fingerprint.pb
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\saved_model.pb
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\variables


## 15. Menjalankan Pipeline dengan Apache Beam

Seluruh komponen di atas dirangkai menjadi sebuah **TFX Pipeline** (`sonnyariady-pipeline/pipeline.py`) dan dijalankan menggunakan **Apache Beam** sebagai *Pipeline Orchestrator* melalui `BeamDagRunner`:

```bash
python sonnyariady-pipeline/run_pipeline.py
```

Artifact hasil eksekusi (metadata MLMD, model, hasil evaluasi) disimpan pada `pipeline_output/tfx_pipeline` dan model serving dipush ke `serving_model/`. Isi folder hasil eksekusi ditampilkan di bawah ini.

In [13]:
# Artifact hasil eksekusi pipeline BeamDagRunner (run_pipeline.py)
pipeline_output_dir = os.path.join(PROJECT_ROOT, "pipeline_output", "tfx_pipeline")
for root, dirs, files in os.walk(pipeline_output_dir):
    dirs[:] = [d for d in dirs if d not in {".temp_dirs", "executor_outputs"}]
    depth = root.replace(pipeline_output_dir, "").count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/")

print()
print("Model serving:")
for path in glob.glob(os.path.join(SERVING_MODEL_DIR, "*")):
    print(" -", path)


Model serving:
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\assets
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\fingerprint.pb
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\saved_model.pb
 - C:\Latihan\AI\Dicoding\TugasAkhirMLOPS\serving_model\variables


## 16. Performa Model

Ringkasan performa model hasil training & evaluasi:

| Metrik | Nilai |
|---|---|
| BinaryAccuracy (evaluasi) | *(lihat output Evaluator di atas)* |
| AUC | *(lihat output Evaluator di atas)* |
| Precision / Recall | *(lihat output Evaluator di atas)* |

Nilai aktual tercetak pada output sel Evaluator setelah notebook dijalankan dan dirangkum pada berkas `README.md`.

## 17. Deployment ke Cloud

Model di-deploy menggunakan **Docker** ke platform **Heroku** (atau alternatif **Railway**):

1. Web app Flask (`app.py`) memuat SavedModel dari `serving_model/`.
2. Endpoint yang tersedia:
   - `GET /` — informasi layanan
   - `GET /health` — health check
   - `POST /predict` — prediksi usia abalon (JSON)
   - `GET /metrics` — metrik Prometheus
3. Build image: `docker build -t sonnyariady-mlops .`
4. Deploy: push image ke Railway / hubungkan repository ke Heroku.

**Tautan web app model serving:** (diisi setelah deployment — contoh: `https://sonnyariady-mlops.up.railway.app`)

Screenshot keberhasilan deployment disimpan dengan nama **`sonnyariady-deployment.png`**.

## 18. Monitoring dengan Prometheus & Grafana

Sistem dipantau menggunakan **Prometheus** yang mengumpulkan metrik dari endpoint `/metrics`:

- `predict_requests_total` — jumlah permintaan prediksi
- `predict_errors_total` — jumlah prediksi yang gagal
- `predict_latency_seconds` — latensi prediksi (histogram)
- metrik proses (CPU, memori, dan sebagainya)

Stack monitoring (lihat folder `monitoring/`):

- `Dockerfile` — image Prometheus
- `prometheus.config` & `prometheus.yml` — konfigurasi scrape
- `docker-compose.yml` — menjalankan app + Prometheus + Grafana sekaligus
- `grafana/` — provisioning datasource & dashboard

Screenshot dashboard monitoring disimpan dengan nama **`sonnyariady-monitoring.png`** (Prometheus) dan **`sonnyariady-grafana-dashboard.png`** (Grafana).

## 19. Kesimpulan

Proyek ini membangun sistem machine learning end-to-end untuk memprediksi kategori usia abalon:

| Tahap | Implementasi |
|---|---|
| Pipeline | TFX (ExampleGen → StatisticsGen → SchemaGen → ExampleValidator → Transform → Tuner → Trainer → Resolver → Evaluator → Pusher) |
| Orchestrator | Apache Beam (BeamDagRunner) |
| Model | DNN (Embedding + Dense), klasifikasi biner |
| Deployment | Flask + Docker → Heroku/Railway |
| Monitoring | Prometheus + Grafana |

Seluruh komponen pipeline disimpan pada folder `sonnyariady-pipeline`, dokumentasi proyek pada `README.md`, dan pengujian prediction request pada `sonnyariady-testing.ipynb`.